In [ ]:
# Shared setup and reusable notebook helpers
import importlib
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import illustris_python as il

root = Path("/public/home/zju_visitor/LiuYuanhao/SAM_project/bymyself")
data_dir = root / "data"
basePath = "/public/share/chenhouzun/TNG50-1/output/"

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

env = os.environ.copy()
env["PYTHONPATH"] = str(root) + os.pathsep + env.get("PYTHONPATH", "")

def run_script(script, *args):
    subprocess.run([sys.executable, str(script), *map(str, args)], cwd=root, env=env, check=True)

def run_snapshots(script, output_template, label, snap_list=snaps):
    for snap in snap_list:
        output_file = data_dir / output_template.format(snap=snap)
        if output_file.is_file():
            print(f"{label} file for snap {snap} already exists, skip.")
            continue
        print(f"Running {label} calculation for snap {snap}")
        run_script(script, snap)
    print(f"All {label} calculations finished.")

def run_plot(script, finished_message):
    run_script(script)
    print(finished_message)


In [ ]:
# Gas parent classification: select inner star-forming or cold gas cells
tree_file = data_dir / "all_selected_trees.hdf5"
data_dir.mkdir(parents=True, exist_ok=True)

if not tree_file.is_file():
    raise FileNotFoundError(f"Selected-tree file does not exist: {tree_file}. Run the selected-tree creation step first.")

import gas_parent_classification as gpc
importlib.reload(gpc)  # load the latest saved version of the module

requested_snaps = [44, 45, 64, 65, 94, 95]
pending_snaps = [snap for snap in requested_snaps if not (data_dir / f"gas_parent_classification_{snap}.hdf5").is_file()]

if pending_snaps:
    print("Running gas-parent classification for snapshots:", pending_snaps)
    for snap in pending_snaps:
        gpc.gas_parent_classification(basePath, snap)
    print("Gas-parent classification finished.")
else:
    print("Gas-parent classification files already exist; skipped all snapshots.")


In [ ]:
# Classify start-snapshot gas and trace its parents to the target snapshots
import gas_parent_classification as gpc
import tracerFuncs as tF
importlib.reload(gpc)
importlib.reload(tF)

tracer_jobs = []
for earlier_snap, later_snap in [(44, 45), (64, 65), (94, 95)]:
    tracer_jobs.extend((later_snap, target_snap) for target_snap in range(later_snap - 5, later_snap))
    tracer_jobs.extend((earlier_snap, target_snap) for target_snap in range(later_snap, later_snap + 5))

required_inputs = [data_dir / "all_selected_trees.hdf5"]
missing_inputs = [path for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError("Missing inputs required by gas-parent classification:\n" + "\n".join(map(str, missing_inputs)))

for tracer_start_snap in sorted({start_snap for start_snap, _ in tracer_jobs}):
    gas_class_file = data_dir / f"gas_parent_classification_{tracer_start_snap}.hdf5"
    if gas_class_file.is_file():
        print(f"Gas-parent classification already exists, reuse: {gas_class_file}")
    else:
        print(f"Classifying start-snapshot gas for snap {tracer_start_snap}", flush=True)
        gpc.gas_parent_classification(basePath, tracer_start_snap)

for tracer_start_snap, tracer_target_snap in tracer_jobs:
    parent_file = data_dir / f"parent_indices_start_{tracer_start_snap}_target_{tracer_target_snap}.hdf5"
    if parent_file.is_file():
        print(f"Tracer-parent file already exists, skip: {parent_file}")
    else:
        print(f"Tracing inner-gas parents: start_snap={tracer_start_snap}, target_snap={tracer_target_snap}", flush=True)
        tF.TraceBackAllInnerGas(basePath, tracer_start_snap, tracer_target_snap)

print("Requested tracer-parent files are ready.")


In [ ]:
# Classify tracer distance and temperature for the requested snapshot pairs
import distance_temp_cats as dtc
importlib.reload(dtc)

distance_jobs = []
for earlier_snap, later_snap in [(44, 45), (64, 65), (94, 95)]:
    distance_jobs.extend((later_snap, target_snap) for target_snap in range(later_snap - 5, later_snap))
    distance_jobs.extend((earlier_snap, target_snap) for target_snap in range(later_snap, later_snap + 5))
distance_output_dir = data_dir / "distance_cats"
distance_output_dir.mkdir(parents=True, exist_ok=True)

pending_jobs = []
for distance_start_snap, distance_target_snap in distance_jobs:
    output_file = distance_output_dir / f"distance_cats_start_{distance_start_snap}_target_{distance_target_snap}.hdf5"
    if output_file.is_file():
        print(f"Distance-temperature file already exists, skip: {output_file}")
        continue
    pending_jobs.append((distance_start_snap, distance_target_snap))

if pending_jobs:
    required_inputs = [data_dir / "all_selected_trees.hdf5", data_dir / "SubhaloPos_new_extrapolated.hdf5", data_dir / "subhaloFlag.hdf5"]
    required_inputs.extend(data_dir / f"gas_parent_classification_{start_snap}.hdf5" for start_snap in sorted({start_snap for start_snap, _ in pending_jobs}))
    required_inputs.extend(data_dir / f"parent_indices_start_{start_snap}_target_{target_snap}.hdf5" for start_snap, target_snap in pending_jobs)
    missing_inputs = [path for path in required_inputs if not path.is_file()]
    if missing_inputs:
        raise FileNotFoundError("Missing inputs required by distance_temp_cats:\n" + "\n".join(map(str, missing_inputs)))

for distance_start_snap, distance_target_snap in pending_jobs:
    print(f"Running distance-temperature classification: start_snap={distance_start_snap}, target_snap={distance_target_snap}", flush=True)
    dtc.distance_temp_cats(basePath, distance_start_snap, distance_target_snap)

print("Requested distance-temperature classifications finished.")


In [ ]:
# Lgal cooling rates for snapshot windows 44/45, 64/65, and 94/95
import cooling_rate_funcs as crf
importlib.reload(crf)

for snap_1, snap_2 in [(44, 45), (64, 65), (94, 95)]:
    lgal_rate_file = data_dir / f"Lgal_cooling_rate_{snap_1}_and_{snap_2}.hdf5"
    if lgal_rate_file.is_file():
        print(f"Lgal cooling-rate file already exists, skip: {lgal_rate_file}")
        continue
    print(f"Calculating Lgal cooling rate: snaps {snap_1}/{snap_2}", flush=True)
    crf.Lgal_cooling_rate(basePath, snap_1=snap_1, snap_2=snap_2)


In [ ]:
# Direct tracer inflow rates: later-snapshot cold gas traced to the earlier snapshot
import cooling_rate_funcs as crf
importlib.reload(crf)

for snap_1, snap_2 in [(44, 45), (64, 65), (94, 95)]:
    direct_in_file = data_dir / "distance_cats" / f"tcrd_{snap_1}_to_{snap_2}.hdf5"
    if direct_in_file.is_file():
        print(f"Direct tracer inflow-rate file already exists, skip: {direct_in_file}")
        continue
    print(f"Calculating direct tracer inflow rate: start_snap={snap_2}, target_snap={snap_1}", flush=True)
    crf.tracer_cooling_rate_direct(basePath, target_snap=snap_1, start_snap=snap_2)


In [ ]:
# Direct tracer outflow rates: earlier-snapshot cold gas traced to the later snapshot
import cooling_rate_funcs as crf
importlib.reload(crf)

for snap_1, snap_2 in [(44, 45), (64, 65), (94, 95)]:
    direct_out_file = data_dir / "distance_cats" / f"tord_{snap_1}_to_{snap_2}.hdf5"
    if direct_out_file.is_file():
        print(f"Direct tracer outflow-rate file already exists, skip: {direct_out_file}")
        continue
    print(f"Calculating direct tracer outflow rate: start_snap={snap_1}, target_snap={snap_2}", flush=True)
    crf.tracer_out_rate_direct(basePath, target_snap=snap_2, start_snap=snap_1)


In [ ]:
# Calculate median profiles for all selected halos in the three mass ranges
import real_physical_properties as rpp
importlib.reload(rpp)

properties_snaps = [40, 67, 99]
for properties_snap in properties_snaps:
    properties_file = data_dir / f"real_physical_properties_{properties_snap}.hdf5"
    if properties_file.is_file():
        print(f"Physical-properties file already exists, skip snap {properties_snap}: {properties_file}")
        continue
    print(f"Calculating real physical properties at snap {properties_snap}", flush=True)
    rpp.real_physical_properties(basePath, snap=properties_snap)
    if not properties_file.is_file():
        raise RuntimeError(f"Physical-properties calculation did not create the expected file: {properties_file}")


In [ ]:
# Calculate radial physical properties for all selected merger-tree halos at plot snapshots
import real_physical_properties_all as rppa
importlib.reload(rppa)

real_properties_snaps = [99, 67, 40]
tree_file = data_dir / "all_selected_trees.hdf5"
if not tree_file.is_file():
    raise FileNotFoundError(f"Selected-tree file does not exist: {tree_file}")

for properties_snap in real_properties_snaps:
    real_properties_file = data_dir / f"real_physical_properties_all_{properties_snap}.hdf5"
    if real_properties_file.is_file():
        print(f"Real physical-properties file already exists, reuse: {real_properties_file}")
    else:
        print(f"Calculating real physical properties for selected-tree halos at snap {properties_snap}", flush=True)
        rppa.real_physical_properties(basePath, snap=properties_snap)
    if not real_properties_file.is_file():
        raise RuntimeError(f"Physical-properties calculation did not create the expected file: {real_properties_file}")


In [ ]:
# Ensure real-profile cooling radii exist for the cooling-radius plot snapshots
import real_cooling_radius as rcr
import real_physical_properties_all as rppa
importlib.reload(rcr)
importlib.reload(rppa)

real_cooling_snaps = [99, 67, 40]
for cooling_snap in real_cooling_snaps:
    real_properties_file = data_dir / f"real_physical_properties_all_{cooling_snap}.hdf5"
    if not real_properties_file.is_file():
        print(f"Missing prerequisite; calculating physical profiles for snap {cooling_snap}", flush=True)
        rppa.real_physical_properties(basePath, snap=cooling_snap)
    if not real_properties_file.is_file():
        raise RuntimeError(f"Physical-properties calculation did not create the expected file: {real_properties_file}")

    real_cooling_file = data_dir / f"real_cooling_radius_{cooling_snap}.hdf5"
    if real_cooling_file.is_file():
        print(f"Real cooling-radius file already exists, reuse: {real_cooling_file}")
    else:
        print(f"Calculating real-profile cooling radius for snap {cooling_snap}", flush=True)
        rcr.real_cooling_radius(cooling_snap, properties_file=real_properties_file)
    if not real_cooling_file.is_file():
        raise RuntimeError(f"Cooling-radius calculation did not create the expected file: {real_cooling_file}")

# Keep the following comparison-plot cell configured for z=0.
real_cooling_snap = 99


In [ ]:
# Calculate the eight z=0 cooling-radius combinations using TNG rho, T, and Z profiles
import real_cooling_radius_replace as rcr_replace
importlib.reload(rcr_replace)

replacement_snap = 99
replacement_properties_file = data_dir / f"real_physical_properties_all_{replacement_snap}.hdf5"
replacement_tree_file = data_dir / "all_selected_trees.hdf5"
replacement_output_file = data_dir / f"real_cooling_radius_replace_{replacement_snap}.hdf5"
required_replacement_inputs = [replacement_properties_file, replacement_tree_file]
missing_replacement_inputs = [path for path in required_replacement_inputs if not path.is_file()]
if missing_replacement_inputs:
    raise FileNotFoundError(f"Missing replacement-cooling-radius inputs: {missing_replacement_inputs}")

if replacement_output_file.is_file():
    print(f"Replacement cooling-radius file already exists, reuse: {replacement_output_file}")
else:
    print(f"Calculating the eight replacement cooling-radius models at snap {replacement_snap}", flush=True)
    rcr_replace.calculate_replacement_cooling_radii(
        snap=replacement_snap,
        properties_file=replacement_properties_file,
        tree_file=replacement_tree_file,
        output_file=replacement_output_file,
    )
if not replacement_output_file.is_file():
    raise RuntimeError(f"Replacement cooling-radius calculation did not create: {replacement_output_file}")
print(f"Replacement cooling-radius calculation is ready: {replacement_output_file}")


In [ ]:
# TNG-measured cooling rates: read existing outputs; otherwise build missing prerequisites and calculate
import h5py
import cooling_rate_funcs as crf
import real_cooling_radius as rcr
import real_physical_properties_all as rppa
importlib.reload(crf)
importlib.reload(rcr)
importlib.reload(rppa)

rate_snap_pairs = [(44, 45), (64, 65), (94, 95)]
tree_file = data_dir / "all_selected_trees.hdf5"
if not tree_file.is_file():
    raise FileNotFoundError(f"Selected-tree file does not exist: {tree_file}")
with h5py.File(tree_file, "r") as f:
    expected_start_ids = np.asarray(f["start_subhalo_id"][:], dtype=np.int64)
    tree_snap_numbers = np.asarray(f["snap_numbers"][:], dtype=np.int64)
    tree_nodes_by_snap = {}
    tree_fofs_by_snap = {}

    for rate_snap in sorted({snap for pair in rate_snap_pairs for snap in pair}):
        snap_columns = np.flatnonzero(tree_snap_numbers == rate_snap)
        if snap_columns.size != 1:
            raise ValueError(f"Expected one selected-tree column for snap {rate_snap}, found {snap_columns.size}.")
        snap_column = int(snap_columns[0])
        tree_nodes_by_snap[rate_snap] = np.asarray(f["mpb_subhalo_ids"][:, snap_column], dtype=np.int64)
        tree_fofs_by_snap[rate_snap] = np.asarray(f["mpb_fof_ids"][:, snap_column], dtype=np.int64)


tng_rate_results = {}
for snap_1, snap_2 in rate_snap_pairs:
    rate_file = data_dir / f"TNG_cooling_rate_{snap_1}_and_{snap_2}.hdf5"
    if rate_file.is_file():
        print(f"TNG cooling-rate file already exists, read and reuse: {rate_file}")
    else:
        # Only missing rate outputs need their physical-profile and measured-radius inputs.
        for rate_snap in (snap_1, snap_2):
            real_properties_file = data_dir / f"real_physical_properties_all_{rate_snap}.hdf5"
            if not real_properties_file.is_file():
                print(f"Missing prerequisite; calculating physical profiles for snap {rate_snap}", flush=True)
                rppa.real_physical_properties(basePath, snap=rate_snap)
            if not real_properties_file.is_file():
                raise RuntimeError(f"Physical-properties calculation did not create the expected file: {real_properties_file}")

            real_cooling_file = data_dir / f"real_cooling_radius_{rate_snap}.hdf5"
            if not real_cooling_file.is_file():
                print(f"Missing prerequisite; calculating real cooling radius for snap {rate_snap}", flush=True)
                rcr.real_cooling_radius(rate_snap, properties_file=real_properties_file)
            if not real_cooling_file.is_file():
                raise RuntimeError(f"Cooling-radius calculation did not create the expected file: {real_cooling_file}")

            with h5py.File(real_cooling_file, "r") as f:
                radius_start_ids = np.asarray(f["start_subhalo_id"][:], dtype=np.int64)
                radius_node_ids = np.asarray(f["subhalo_id"][:], dtype=np.int64)
                radius_fof_ids = np.asarray(f["fof_id"][:], dtype=np.int64)
            if not (np.array_equal(radius_start_ids, expected_start_ids)
                    and np.array_equal(radius_node_ids, tree_nodes_by_snap[rate_snap])
                    and np.array_equal(radius_fof_ids, tree_fofs_by_snap[rate_snap])):
                raise ValueError(f"Real cooling-radius rows do not match the selected tree at snap {rate_snap}: {real_cooling_file}")

        print(f"Calculating TNG-measured cooling rate for snaps {snap_1}/{snap_2}", flush=True)
        crf.TNG_cooling_rate(basePath, snap_1=snap_1, snap_2=snap_2)
    if not rate_file.is_file():
        raise RuntimeError(f"TNG cooling-rate calculation did not create the expected file: {rate_file}")

    with h5py.File(rate_file, "r") as f:
        result_start_ids = np.asarray(f["start_subhalo_id"][:], dtype=np.int64)
        result_snaps = np.asarray(f["snap_numbers"][:], dtype=np.int64)
        if not np.array_equal(result_start_ids, expected_start_ids):
            raise ValueError(f"TNG cooling-rate rows do not match all_selected_trees.hdf5: {rate_file}")
        if not np.array_equal(result_snaps, np.asarray([snap_1, snap_2], dtype=np.int64)):
            raise ValueError(f"Snapshot metadata do not match the requested pair: {rate_file}")
        tng_rate_results[(snap_1, snap_2)] = {
            "halo_mass": np.asarray(f["halo_mass"][:], dtype=np.float64),
            "cooling_rate": np.asarray(f["cooling_rate"][:], dtype=np.float64),
            "valid": np.asarray(f["valid"][:], dtype=bool),
        }
    print(f"Loaded TNG cooling rate for snaps {snap_1}/{snap_2}: {np.count_nonzero(tng_rate_results[(snap_1, snap_2)]['valid']):,} valid trees", flush=True)


In [ ]:
# Calculate BH and entropy properties for the selected z=0 tree halos
import AGN_properties as agnp
importlib.reload(agnp)

agn_properties_snap = 99
agn_properties_output, _, _, _ = agnp.calculate_agn_properties(basePath, snap=agn_properties_snap)
print(f"AGN-property calculation finished: {agn_properties_output}", flush=True)


In [ ]:
# Select three halos with major mergers from snaps 42–99 and calculate both cooling-radius histories
import event_object as eo
importlib.reload(eo)

event_snap_min = 42
event_snap_max = 99
event_selection_file, event_tree_rows, event_start_ids = eo.select_event_halo(basePath, event_snap_min, event_snap_max)
print(f"Selected event halos: {event_start_ids}; saved to {event_selection_file}", flush=True)

event_cooling_file, event_tng_radius, event_lgal_radius = eo.object_cooling_radius(basePath)
print(f"Event-halo cooling-radius calculation finished: {event_cooling_file}", flush=True)


In [ ]:
# Select the sample halos' hot gas at snap 98
from trace_hot_gas import gas_parent_classification_hot as gpch
importlib.reload(gpch)
hot_start_snap = 98
hot_target_snap = 99
hot_gas_classification_file = data_dir / f"gas_parent_classification_hot_{hot_start_snap}.hdf5"
if not (data_dir / "all_selected_trees.hdf5").is_file():
    raise FileNotFoundError("Create all_selected_trees.hdf5 before classifying hot gas.")
if hot_gas_classification_file.is_file():
    print(f"Hot-gas classification already exists, reuse: {hot_gas_classification_file}")
else:
    gpch.gas_parent_classification(basePath, [hot_start_snap])


In [ ]:
# Trace snap-98 hot-gas tracers forward to snap 99
from trace_hot_gas import tracerFuncs_hot as tfh
importlib.reload(tfh)
hot_tracer_parent_file = data_dir / f"parent_indices_hot_start_{hot_start_snap}_target_{hot_target_snap}.hdf5"
if not hot_gas_classification_file.is_file():
    raise FileNotFoundError(f"Run hot-gas classification first: {hot_gas_classification_file}")
if hot_tracer_parent_file.is_file():
    print(f"Hot-gas tracer-parent file already exists, reuse: {hot_tracer_parent_file}")
else:
    tfh.TraceBackAllHotGas(basePath, hot_start_snap, hot_target_snap)


In [ ]:
# Classify snap-99 parents, temperature, and position for the traced hot-gas tracers
from trace_hot_gas import distance_temp_cats_hot as hot_dtc
importlib.reload(hot_dtc)
hot_distance_dir = data_dir / "distance_cats"
hot_distance_dir.mkdir(parents=True, exist_ok=True)
hot_distance_file = hot_distance_dir / f"distance_cats_hot_start_{hot_start_snap}_target_{hot_target_snap}.hdf5"
required_hot_distance_inputs = [data_dir / "all_selected_trees.hdf5", hot_gas_classification_file, hot_tracer_parent_file]
missing_hot_distance_inputs = [path for path in required_hot_distance_inputs if not path.is_file()]
if missing_hot_distance_inputs:
    raise FileNotFoundError("Missing inputs required by distance_temp_cats_hot:\n" + "\n".join(map(str, missing_hot_distance_inputs)))
if hot_distance_file.is_file():
    print(f"Hot-gas distance-temperature file already exists, reuse: {hot_distance_file}")
else:
    hot_dtc.distance_temp_cats(basePath, hot_start_snap, hot_target_snap)


In [ ]:
# Calculate radial hot-to-cold tracer fractions for MW-like sample halos
from trace_hot_gas import hot_to_cold_fraction as htc
importlib.reload(htc)
hot_fraction_file = hot_distance_dir / f"hot_to_cold_fraction_MW_start_{hot_start_snap}_target_{hot_target_snap}.hdf5"
required_hot_fraction_inputs = [data_dir / "all_selected_trees.hdf5", hot_gas_classification_file, hot_tracer_parent_file, hot_distance_file]
missing_hot_fraction_inputs = [path for path in required_hot_fraction_inputs if not path.is_file()]
if missing_hot_fraction_inputs:
    raise FileNotFoundError("Missing inputs required by hot_to_cold_fraction:\n" + "\n".join(map(str, missing_hot_fraction_inputs)))
if hot_fraction_file.is_file():
    print(f"Hot-to-cold fraction file already exists, reuse: {hot_fraction_file}")
else:
    hot_fraction_file, hot_fraction_by_halo = htc.hot_to_cold_fraction(basePath, hot_start_snap, hot_target_snap)
    print(f"Hot-to-cold fraction calculation finished: {hot_fraction_file}", flush=True)
